# Your First FastAPI App

This notebook covers:

1. Define a FastAPI app and a handful of endpoints
2. Run requests against it with `TestClient` — no server needed
3. Read the auto-generated OpenAPI / Swagger UI docs
4. Recognize what FastAPI gives you for free

**Scope**: `fastapi`, `httpx` (via `TestClient`), a touch of `pydantic` (full treatment in notebook 1.3).

We'll use the portfolio / asset analytics domain throughout the series — `Asset(ticker, name, price)` will appear again in every chapter and ends up in the capstone unchanged.

## 1. Installing FastAPI

FastAPI is on PyPI as `fastapi`. To actually serve traffic you also want an ASGI server — `uvicorn` is the standard pick. Both are already pinned in `requirements.txt` at the repo root:

```text
fastapi>=0.108.0
uvicorn[standard]>=0.25.0
```

For this notebook we don't need to *run* uvicorn — `TestClient` calls the app in-process. Uvicorn comes back in chapter 8 when we look at workers and lifecycle.

Verify the install:

In [ ]:
import fastapi
import pydantic

print("fastapi :", fastapi.__version__)
print("pydantic:", pydantic.__version__)

## 2. The Smallest Possible App

A FastAPI app is an object plus a few decorated functions. That's it — no class to subclass, no config file required.

The decorator `@app.get("/")` registers the function as the handler for `GET /`. Whatever you return gets JSON-encoded automatically (dicts, lists, Pydantic models, even dataclasses).

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Portfolio API", version="0.1.0")

@app.get("/")
def read_root():
    return {"service": "portfolio-api", "status": "ok"}

# TestClient runs the app in-process — no socket, no separate server.
client = TestClient(app)

resp = client.get("/")
print("status:", resp.status_code)
print("body  :", resp.json())

## 3. Adding More Endpoints

Three things to introduce here, all of which FastAPI makes painless:

- **Path parameters** — `{ticker}` in the URL becomes a typed function argument.
- **Query parameters** — function arguments that aren't in the path become query string params.
- **Request bodies** — typed via a Pydantic model. Validation, docs, and the OpenAPI schema all come from this one declaration. (Notebook 1.3 goes deep on Pydantic; here we just use it.)

We'll also use `status_code=` to return `201 Created` on creation, and `tags=` to group endpoints in the docs.

In [ ]:
from fastapi import status
from pydantic import BaseModel

class AssetIn(BaseModel):
    ticker: str
    name: str
    price: float

# Pretend in-memory store, just to make the example real.
ASSETS: dict[str, dict] = {
    "AAPL": {"ticker": "AAPL", "name": "Apple Inc.", "price": 195.0},
    "MSFT": {"ticker": "MSFT", "name": "Microsoft Corp.", "price": 410.0},
}

@app.get("/assets", tags=["assets"], summary="List assets")
def list_assets(limit: int = 10):
    return list(ASSETS.values())[:limit]

@app.get("/assets/{ticker}", tags=["assets"], summary="Get one asset by ticker")
def get_asset(ticker: str):
    asset = ASSETS.get(ticker.upper())
    if asset is None:
        # We'll switch to HTTPException + handlers in chapter 6 — for now, a stub 404.
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail=f"unknown ticker: {ticker}")
    return asset

@app.post("/assets", tags=["assets"], status_code=status.HTTP_201_CREATED, summary="Create asset")
def create_asset(asset: AssetIn):
    record = asset.model_dump() | {"ticker": asset.ticker.upper()}
    ASSETS[record["ticker"]] = record
    return record

print("registered routes:")
for r in app.routes:
    if hasattr(r, "methods"):
        print(f"  {','.join(r.methods):10} {r.path}")

## 4. The `TestClient` Workflow

`TestClient` is `httpx` wired to call your FastAPI app directly — no network, no port. That makes it perfect for unit tests *and* for prose-style exploration like this notebook.

The shape will be familiar from notebook 1.1: `.get`, `.post`, `.json()`, `.status_code`. The difference is that responses come back instantly because there is no socket in between.

In [ ]:
# Happy path: list + read
print("GET /assets       ->", client.get("/assets").status_code, client.get("/assets").json())
print("GET /assets/AAPL  ->", client.get("/assets/AAPL").status_code, client.get("/assets/AAPL").json())

# Query parameter
print("GET /assets?limit=1 ->", client.get("/assets", params={"limit": 1}).json())

# Sad path: 404
r = client.get("/assets/ZZZZ")
print("GET /assets/ZZZZ  ->", r.status_code, r.json())

# Create — note the 201 status code
r = client.post("/assets", json={"ticker": "googl", "name": "Alphabet Inc.", "price": 175.0})
print("POST /assets      ->", r.status_code, r.json())

# Validation failure — 422 comes for free from Pydantic
r = client.post("/assets", json={"ticker": "NVDA"})  # missing name + price
print("POST /assets (bad)->", r.status_code)
print("  errors:", r.json()["detail"])

## 5. Auto-Generated OpenAPI Schema

Every FastAPI app exposes its OpenAPI 3.x schema at `/openapi.json`. This is generated from your type hints and Pydantic models — you don't write it by hand and it can't drift out of sync.

The schema is what powers `/docs` (Swagger UI) and `/redoc` (ReDoc). It's also what tools like client-code generators and Postman read.

In [ ]:
schema = client.get("/openapi.json").json()

print("openapi version:", schema["openapi"])
print("info           :", schema["info"])

print("\npaths:")
for path, ops in schema["paths"].items():
    for method in ops:
        op = ops[method]
        print(f"  {method.upper():5} {path:18} tags={op.get('tags')}  summary={op.get('summary')!r}")

print("\nrequest body schema for POST /assets:")
post_body = schema["paths"]["/assets"]["post"]["requestBody"]
print(" ", post_body["content"]["application/json"]["schema"])

print("\ncomponents.schemas:")
for name, defn in schema["components"]["schemas"].items():
    print(f"  {name}: {list(defn.get('properties', {}).keys())}")

## 6. Swagger UI and ReDoc

The same schema drives two interactive docs UIs that FastAPI mounts for free:

- **`/docs`** — Swagger UI. Lets you call endpoints from the browser, with auto-generated forms for path/query/body inputs.
- **`/redoc`** — ReDoc. Read-only, single-page, friendlier for reference reading.

`TestClient` calls don't render these — they're rendered by browsers. To see them, run the app with uvicorn from a terminal:

```bash
uvicorn 01_rest_foundations.app:app --reload   # if the app is in a .py file
```

For now, just know they exist and that they update the instant you change a type hint or add a route. Notebook 8.1 covers running uvicorn properly (workers, lifespan, reload tradeoffs).

## Key Takeaways

- A FastAPI app is one `FastAPI()` instance plus decorated handlers. No subclassing, no boilerplate.
- Type hints aren't just for the IDE — they drive runtime validation (Pydantic), the OpenAPI schema, and the docs UIs.
- `TestClient` runs the app in-process. Use it for tests *and* for notebooks like this one — never spin up a real server for unit-level work.
- `/openapi.json`, `/docs`, and `/redoc` are generated, not authored. If the schema is wrong, fix the type hints — don't edit the JSON.
- `status_code=`, `tags=`, `summary=` on the route decorator shape both behavior and documentation in one place.

Next: notebook 1.3 — Pydantic validation in depth. The `AssetIn` model used here is the simplest possible case; chapter 2 and beyond rely on much richer validation.

## Exercises

**1. Add a `/health` endpoint.** It should:

- Respond to `GET /health`.
- Return `{"status": "ok"}` with status code `200`.
- Be tagged `infra` and have summary `"Liveness probe"`.
- Show up in `/openapi.json` under the `infra` tag.

**2. Modify the response description.** Add `response_description="The newly created asset"` to the `POST /assets` decorator. Then fetch `/openapi.json` and confirm the description appears under the `201` response in the schema.

**3. Validation behavior.** Without changing `AssetIn`, find a body that gets a `422` and another body that gets a `200/201`. Then change one field on `AssetIn` to `int` and explain why a previously-passing body now fails.